# 01 — Test market-data (état container + healthcheck + env vars)

Smoke test du container `fxvol-market-data` — étape 1/5. Valide les **fondations** avant de tester la chaîne Redis et WS dans les notebooks suivants : container UP, healthcheck heartbeat-based passe healthy, env vars correctement injectées au boot.

**Couvre** :
1. Container présent et `running` (pas `exited`, `restarting`, `dead`)
2. Docker healthcheck = `healthy` (= heartbeat:market_data en Redis frais < 60s, cf. `docker-compose.yml § market-data`)
3. Image attendue (`fx-options-market-data:local`) + `StartedAt` + restart count
4. Env vars critiques injectées correctement : `SERVICE_NAME=market_data`, `IB_HOST=ib-gateway`, `IB_PORT=4002`, `IB_CLIENT_ID=1`, `MARKET_SYMBOL=EURUSD`, `REDIS_URL=redis://redis:6379/0`

**Préreq** :
- Stack démarrée avec profiles `engines` + `ib` : `docker compose --profile engines --profile ib up -d` (ou `.\scripts\start_stack.ps1` qui le fait pour toi)
- Container `redis` healthy (le healthcheck market-data dépend de Redis pour lire `heartbeat:market_data`)
- Container `ib-gateway` healthy (sinon market-data ne peut pas pull les ticks → pas de heartbeat → healthcheck unhealthy)
- ~30-60s d'attente après le `up` pour que le start_period soit fini et qu'IBC + market-data aient eu le temps de pousser le 1er heartbeat

**Référence** : `docker-compose.yml § market-data`, `src/services/market_data/main.py` (entrypoint), `src/services/market_data/engine.py` (loop).

## Setup — helpers

On utilise `subprocess.run` pour interroger Docker. Aucun secret manipulé ici, mais on respecte la règle CLAUDE.md `feedback_docker_inspect_env_leak` : les inspections d'env vars passent par un **filtre positif explicite** côté `docker exec env`, jamais de dump complet via `docker inspect Config.Env`.

In [33]:
import subprocess

CONTAINER = "fxvol-market-data"
results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

def docker_inspect(fmt):
    out = subprocess.run(
        ["docker", "inspect", "-f", fmt, CONTAINER],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

print(f"target container = {CONTAINER}")

target container = fxvol-market-data


## 1. Container présent et `running`

**Ce que tu dois voir** : state = `running`. Si tu vois `exited` → le container a crashé (logs avec `docker logs fxvol-market-data --tail 100`). Si `restarting` → boucle de crash, regarder aussi les logs. Si `<not found>` → la stack n'a pas été lancée avec le profile `engines`.

In [34]:
print("== 1. container state ==")

state = docker_inspect("{{.State.Status}}")
record("docker container state", state == "running", state or "<not found>")

restart_count = docker_inspect("{{.RestartCount}}")
# Restart count > 0 = signal de crashloop ; > 5 = sérieusement cassé
record("restart count raisonnable (≤ 3)",
       restart_count.isdigit() and int(restart_count) <= 3,
       f"restart count = {restart_count}")

== 1. container state ==
  [OK  ] docker container state  | running
  [OK  ] restart count raisonnable (≤ 3)  | restart count = 0


 ## 2. Docker healthcheck = `healthy`

Le healthcheck du `docker-compose.yml § market-data` est un script Python qui lit `heartbeat:market_data` dans Redis et vérifie que le timestamp est < 60s. Donc :

- `healthy` → market-data est bien connecté à IB ET à Redis, et il pousse activement des heartbeats
- `unhealthy` → soit IB Gateway down (pas de ticks → pas de heartbeat), soit Redis down (le healthcheck lui-même fail), soit le service crash silencieusement entre 2 heartbeats
- `starting` → on est dans le `start_period` (30s), normal au boot, attendre
- `<no healthcheck>` → le healthcheck n'est pas configuré (= compose désaligné avec ce qu'on attend)

In [35]:
print("== 2. healthcheck ==")

health = docker_inspect("{{.State.Health.Status}}")
record("docker healthcheck", health == "healthy", health or "<no healthcheck>")

# Pour debug : dernière sortie du healthcheck script (utile si unhealthy)
last_log = docker_inspect("{{(index .State.Health.Log 0).Output}}")
if last_log:
    print(f"  [INFO] dernière sortie healthcheck : {last_log[:200]!r}")

== 2. healthcheck ==
  [OK  ] docker healthcheck  | healthy


## 3. Image + StartedAt (info)

Pas un test pass/fail — juste de l'info pour le diag. On veut voir l'image utilisée et depuis quand le container tourne.

In [36]:
print("== 3. image + uptime info ==")

image = docker_inspect("{{.Config.Image}}")
started_at = docker_inspect("{{.State.StartedAt}}")
print(f"  [INFO] image      : {image}")
print(f"  [INFO] StartedAt  : {started_at}")

# Optionnel : sanity check sur le nom d'image attendu (compose default)
expected_prefix = "fx-options-market-data"
record(f"image préfixe '{expected_prefix}'",
       expected_prefix in image,
       image)

== 3. image + uptime info ==
  [INFO] image      : fx-options-market-data:local
  [INFO] StartedAt  : 2026-04-29T09:02:03.701821295Z
  [OK  ] image préfixe 'fx-options-market-data'  | fx-options-market-data:local


## 4. Env vars critiques injectées

**Filtre positif explicite** (cf. mémoire projet `feedback_docker_inspect_env_leak`) : on liste seulement les vars qui nous intéressent et qui ne sont **jamais des secrets**. Pas de dump complet `docker inspect Config.Env`.

**Ce que tu dois voir** : les 6 vars présentes avec leurs valeurs attendues. Si une diffère, le compose a probablement été modifié (ou un override applique des valeurs custom).

In [37]:
print("== 4. env vars critiques (filtrées non-secret) ==")

expected = {
    "SERVICE_NAME": "market_data",
    "IB_HOST": "ib-gateway",
    "IB_PORT": "4002",
    "IB_CLIENT_ID": "1",
    "MARKET_SYMBOL": "EURUSD",
    "REDIS_URL": "redis://redis:6379/0",
}

# docker exec env | grep filter — pattern positif explicite, pas de wildcard.
key_pattern = "^(" + "|".join(expected) + ")="
out = subprocess.run(
    ["docker", "exec", CONTAINER, "sh", "-c",
     f'env | grep -E "{key_pattern}"'],
    capture_output=True, text=True,
)
lines = out.stdout.strip().splitlines()
actual = dict(line.split("=", 1) for line in lines if "=" in line)

for key, expected_val in expected.items():
    actual_val = actual.get(key, "<missing>")
    record(f"{key} = {expected_val!r}",
           actual_val == expected_val,
           actual_val if actual_val != expected_val else "")

== 4. env vars critiques (filtrées non-secret) ==
  [OK  ] SERVICE_NAME = 'market_data'
  [OK  ] IB_HOST = 'ib-gateway'
  [OK  ] IB_PORT = '4002'
  [OK  ] IB_CLIENT_ID = '1'
  [OK  ] MARKET_SYMBOL = 'EURUSD'
  [OK  ] REDIS_URL = 'redis://redis:6379/0'


## Récap final

In [38]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 100)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 100)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK container market-data sain. Pass au notebook 02 (Redis outputs).")


LABEL                                                   STATUS  DETAIL
----------------------------------------------------------------------------------------------------
docker container state                                  OK      running
restart count raisonnable (≤ 3)                         OK      restart count = 0
docker healthcheck                                      OK      healthy
image préfixe 'fx-options-market-data'                  OK      fx-options-market-data:local
SERVICE_NAME = 'market_data'                            OK      
IB_HOST = 'ib-gateway'                                  OK      
IB_PORT = '4002'                                        OK      
IB_CLIENT_ID = '1'                                      OK      
MARKET_SYMBOL = 'EURUSD'                                OK      
REDIS_URL = 'redis://redis:6379/0'                      OK      
----------------------------------------------------------------------------------------------------

10 OK / 0 FAIL  

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `docker container state` = `<not found>` | Stack lancée sans profile `engines` | `docker compose --profile engines --profile ib up -d` |
| `state` = `restarting` ou restart count > 3 | Crashloop : engine plante au boot | `docker logs fxvol-market-data --tail 100` ; suspects fréquents : `IB_HOST` non résolvable, IB Gateway pas encore healthy |
| `healthcheck` = `starting` depuis > 60s | start_period dépassé, heartbeat n'arrive pas | Vérifier IB Gateway healthy (sans IB ticks → pas de heartbeat) ; sinon `docker logs --tail 50 fxvol-market-data \| grep -i heartbeat` |
| `healthcheck` = `unhealthy` | Heartbeat absent ou stale > 60s | `docker compose exec redis redis-cli GET heartbeat:market_data` ; si vide ou TTL négatif → engine ne pousse plus |
| `IB_HOST != "ib-gateway"` | Override compose appliqué | Vérifier `.env` ou `docker-compose.override.yml` — un override prend la priorité sur les valeurs du compose principal |
| `IB_CLIENT_ID != "1"` | Override pour faire cohabiter avec un autre engine | OK si voulu, sinon revoir compose |
| `image` ≠ `fx-options-market-data:*` | Image custom (release pinnée, registry distant) | Pas un fail si voulu, sinon revoir build |
| `MARKET_SYMBOL` ≠ `EURUSD` | Override pour tester un autre symbole | OK si voulu (le smoke est conçu pour EURUSD, à adapter sinon) |